In [14]:
# Python 3.10.11
%pip install -r requirements.txt > /dev/null
from args import *

Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # 检查GPU

# TODO(241225) 依赖导入
import pandas as pd

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# 构造含交叉注意力机制的Transformer
class TransformerEncoderLayerWithCrossAttention(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super(TransformerEncoderLayerWithCrossAttention, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        # 自注意力机制
        src2 = self.self_attn(src, src, src, attn_mask=src_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # 将输入序列平均拆分为4等分
        seq_len = src.size(1)
        seg_len = seq_len // 4
        seg1 = src[:, :seg_len, :]
        seg2 = src[:, seg_len:2*seg_len, :]
        seg3 = src[:, 2*seg_len:3*seg_len, :]
        seg4 = src[:, 3*seg_len:, :]

        # 交叉注意力机制
        seg1_cross, _ = self.cross_attn(seg1, seg2, seg2)
        seg2_cross, _ = self.cross_attn(seg2, seg3, seg3)
        seg3_cross, _ = self.cross_attn(seg3, seg4, seg4)
        seg4_cross, _ = self.cross_attn(seg4, seg1, seg1)

        # 合并交叉注意力结果
        src_cross = torch.cat([seg1_cross, seg2_cross, seg3_cross, seg4_cross], dim=1)
        src = src + self.dropout2(src_cross)
        src = self.norm2(src)

        # 线性层和残差连接
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = src + self.dropout3(src2)
        src = self.norm3(src)

        return src

class TransformerEncoderWithCrossAttention(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super(TransformerEncoderWithCrossAttention, self).__init__()
        self.layers = nn.ModuleList([encoder_layer for _ in range(num_layers)])
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src, mask=None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask)

        if self.norm is not None:
            output = self.norm(output)

        return output


# 定义模型结构
class MultiOmicsModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(MultiOmicsModel, self).__init__()

        # num_features = 80  # 特征数量
        global num_features

        # 共享隐藏层
        self.shared_hidden = nn.Linear(num_features * 2, 128)  # 4个组学，每个80维度
        
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(num_features, 64)

        # 基因表达独立隐藏层
        self.gene_hidden_layers = nn.Sequential(nn.Linear(128, 64))
        # 拷贝数数据的独立隐藏层
        self.cnv_hidden_layers = nn.Sequential(nn.Linear(128, 64))
        

        # 创建一个带有交叉注意力的Transformer编码器层
        d_model = 64
        nhead = 8
        dim_feedforward = 512
        dropout = 0.1
        num_layers = 6
        encoder_layer = TransformerEncoderLayerWithCrossAttention(d_model, nhead, dim_feedforward, dropout)
        # 创建一个带有交叉注意力的Transformer编码器
        self.encoder = TransformerEncoderWithCrossAttention(encoder_layer, num_layers)

        # 输出层
        self.lin = nn.Linear(64, 1)

        # ReLU 激活函数
        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, expr, cnv, report, sis):
        # 共享隐藏层的前向传播
        x = torch.concat([expr, cnv], dim=1)
        x = self.relu(self.shared_hidden(x))

        x_report = self.fc1(report)
        x_report = self.relu(x_report)

        x_sis = self.fc2(sis)
        x_sis = self.relu(x_sis)

        x = self.dropout(x)

        # 基因表达数据的独立隐藏层的前向传播
        x_gene = self.gene_hidden_layers(x)
        x_gene = self.relu(x_gene)
        # 拷贝数数据的独立隐藏层的前向传播
        x_cnv = self.cnv_hidden_layers(x)
        x_cnv = self.relu(x_cnv)
       
        # 合并多组学
        x = torch.stack([x_gene, x_cnv, x_report, x_sis])
        x = self.dropout(x)

        # transformer层
        x = x.permute(1,0,2)
        x = self.encoder(x)
        x = x.permute(1,0,2)
        x = self.relu(x)
        x = self.dropout(x)
        
        # 输出层
        x = x[0]
        out = self.lin(x)

        return out

# 定义Dataset对象
class MultiOmicsDataset(Dataset):
    def __init__(self, expr, cnv, report, sis, label):
        self.expr = torch.tensor(expr, dtype=torch.float32)
        self.cnv = torch.tensor(cnv, dtype=torch.float32) 
        self.report = torch.tensor(report, dtype=torch.float32)
        self.sis = torch.tensor(sis, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.float32)

    def __len__(self):
        return len(self.label)

    def __getitem__(self, index):
        self.expr[index]

        return self.expr[index], self.cnv[index], self.report[index], self.sis[index], self.label[index]


In [17]:
## ## ## ## ## ## ## ## 分割线 ## ## ## ## ## ## ## ##

In [18]:
# 调试代码
num_samples = 312  # 样本数量
num_features = 768  # 特征数量

gene_data = torch.zeros(num_samples, num_features) # 临时占位

In [19]:
# TODO(241225) 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)
sample_key_list = label_df.index.to_list()
print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
gene_df = load_gene_data_by_sample_key(sample_key_list)
cnv_df = load_cnv_data_by_sample_key(sample_key_list)

wsi_array = load_wsi_data_by_sample_key(sample_key_list)
report_array = load_report_data_by_sample_key(sample_key_list)

label 样本数量: 312


In [20]:
# assert 0, "Debug Point"
expr = torch.from_numpy(gene_df.values[:num_samples, :num_features])
label = torch.from_numpy(label_df.values[:num_samples]).reshape(-1)

cnv = torch.from_numpy(cnv_df.values[:num_samples, :num_features])
report = torch.from_numpy(report_array[:num_samples, :num_features])
sis = torch.from_numpy(wsi_array[:num_samples, :num_features])

In [21]:
expr.shape

torch.Size([312, 768])

In [22]:
## ## ## ## ## ## ## ## 分割线 ## ## ## ## ## ## ## ##

In [23]:
# 模型训练批次大小
batchsize = 64

# 定义一些超参数
learning_rate = 0.001
num_epochs = 20

train_dataset = MultiOmicsDataset(expr, cnv, report, sis, label)
train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True, num_workers=3, drop_last=True)

/tmp/ipykernel_3838/1882096740.py:153: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.expr = torch.tensor(expr, dtype=torch.float32)
/tmp/ipykernel_3838/1882096740.py:154: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.cnv = torch.tensor(cnv, dtype=torch.float32)
/tmp/ipykernel_3838/1882096740.py:155: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.report = torch.tensor(report, dtype=torch.float32)
/tmp/ipykernel_3838/1882096740.py:156: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().

In [24]:
model = MultiOmicsModel()
# model(expr, cnv, report, sis)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [25]:
def run_fold(fold):    
    for epoch in range(num_epochs):

        # Epoch start

        # Training
        model.train()
        for exper, cnv, report, sis, label in train_loader:
            outputs = model(exper, cnv, report, sis)
            loss = criterion(torch.flatten(outputs), label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # 在训练集上验证模型
        model.eval()  # 设置模型为评估模式
        train_loss = 0
        predictions = [] # 保存预测结果
        true_labels = [] # 保存真实结果
        scores = []      # 保存预测得分
        with torch.no_grad():
            for expr, cnv, report, sis, label in train_loader:
                outputs = model(expr, cnv, report, sis)
                outputs = torch.flatten(outputs)
                train_loss += criterion(outputs, label).item()
                
                predicted = torch.round(torch.sigmoid(outputs))
                predictions.extend(predicted.tolist())
                true_labels.extend(label.tolist())
                scores.extend(outputs.tolist())
        train_loss /= len(train_loader.dataset)
        accuracy = accuracy_score(true_labels, predictions)

        # accuracy_score (真实标签 vs. 预测标签) 计算准确率
        # roc_auc_score  (真实标签 vs. 预测得分) 计算ROC曲线下的面积
        accuracy = accuracy_score(true_labels, predictions)
        auc = roc_auc_score(true_labels, scores)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss}, Train Accuracy: {accuracy:.4f}, Train AUC: {auc:.4f}')
        with open(f'{data_output_dir_path}/log_{fold}.txt', 'a') as FO:
            FO.writelines(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss}, Train Accuracy: {accuracy:.4f}, Train AUC: {auc:.4f}\n')
        
        # 在测试集上验证模型
        model.eval()  # 设置模型为评估模式
        test_loss = 0
        predictions = []
        true_labels = []
        scores = []
        with torch.no_grad():
            for expr, cnv, report, sis, label in train_loader:
                outputs = model(expr, cnv, report, sis)
                outputs = torch.flatten(outputs)
                test_loss += criterion(outputs, label).item()
                predicted = torch.round(torch.sigmoid(outputs)) 
                predictions.extend(predicted.tolist())
                true_labels.extend(label.tolist())
                scores.extend(outputs.tolist())
        # 打印训练过程中的损失和测试精度
        test_loss /= len(train_loader.dataset)
        accuracy = accuracy_score(true_labels, predictions)
        auc = roc_auc_score(true_labels, scores)
        print(f'Epoch [{epoch+1}/{num_epochs}], Test Loss: {test_loss}, Test Accuracy: {accuracy:.4f}, Test AUC: {auc:.4f}\n')
        with open(f'{data_output_dir_path}/log_{fold}.txt', 'a') as FO:
            FO.writelines(f'Epoch [{epoch+1}/{num_epochs}], Test Loss: {test_loss}, Test Accuracy: {accuracy:.4f}, Test AUC: {auc:.4f}\n')

In [26]:
# 实例化五折交叉验证对象
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
fold = 0  # 折数
for train_idx, test_idx in cv.split(expr, label):
    print(f'fold: {fold}')
    fold += 1
    print(f"Train index length: {len(train_idx)}")
    print(f"Test index length: {len(test_idx)}")
    run_fold(fold)

fold: 0
Train index length: 156
Test index length: 156
Epoch [1/20], Train Loss: 0.006617463647555082, Train Accuracy: 0.7891, Train AUC: 0.5381
Epoch [1/20], Test Loss: 0.0064931481312482785, Test Accuracy: 0.7969, Test AUC: 0.6182

Epoch [2/20], Train Loss: 0.006581158496630497, Train Accuracy: 0.7930, Train AUC: 0.4368
Epoch [2/20], Test Loss: 0.006580161742674999, Test Accuracy: 0.7930, Test AUC: 0.4871

Epoch [3/20], Train Loss: 0.006190682928531597, Train Accuracy: 0.8125, Train AUC: 0.4639
Epoch [3/20], Test Loss: 0.0064017488979376275, Test Accuracy: 0.8008, Test AUC: 0.4473

Epoch [4/20], Train Loss: 0.006542885055144628, Train Accuracy: 0.7930, Train AUC: 0.5029
Epoch [4/20], Test Loss: 0.0064722240353241945, Test Accuracy: 0.7969, Test AUC: 0.5372

Epoch [5/20], Train Loss: 0.006540833375392816, Train Accuracy: 0.7930, Train AUC: 0.5020
Epoch [5/20], Test Loss: 0.0066055776790166516, Test Accuracy: 0.7891, Test AUC: 0.5639

Epoch [6/20], Train Loss: 0.006470713095787244, Tra